# Parser Comparison

Decides which PDF parser `src/ingestion/parser.py` uses.

**Criteria, in priority order:**

1. **Nature of the PDF** — born-digital or scanned? Decides whether OCR is in play at all.
2. **Task requirements** — based on the information of the requirements. For example, three of the five Part 1 fields come off Table 1.1 on page 8. A number must bind to the correct *column header*, so row integrity is the deciding property.
3. **LangChain / LangGraph compatibility** — confirms nothing is ruled out for Part 3. Not a discriminator: every candidate has a loader returning the same `Document` objects.

**List of loaders to be considered**

| Loader | Wraps | Strengths | Watch for |
|---|---|---|---|
| `PyMuPDFLoader` | PyMuPDF | Fastest; page metadata; can export images | May fragment table rows |
| `PyPDFLoader` | pypdf | Pure-Python, no binary wheel | Text only, no cell structure |
| `PDFPlumberLoader` | pdfplumber | `extract_table()` returns a real cell grid | Slowest |
| Docling | — | Table/layout/figure understanding | Heavy; advantage lost on a clean text layer |
| OCR | — | — | Scanned PDFs only |

In [14]:
#loading of PDF into repo (note that this is gitignored)
from pathlib import Path

PDF = Path("../data/fy2024_analysis_of_revenue_and_expenditure.pdf")
assert PDF.exists(), f"missing: {PDF.resolve()}"

# 1-indexed, as cited in the task spec
PAGES = [5, 6, 8, 20]
print(PDF.resolve(), f"{PDF.stat().st_size / 1024:.0f} KB")

/Users/cynthialim/structural-extanalyzer/data/fy2024_analysis_of_revenue_and_expenditure.pdf 679 KB


## Criterion 1 — nature of the PDF

Is there a real text layer, or only pixels? If fonts are present and no image-codec
filters are used for page content, the text is extractable and OCR would be strictly
worse — rasterising real characters and guessing them back.

In [ ]:
#check on nature of source, whether it is digital or scanned. For scanned data, OCR will be required.
raw = PDF.read_bytes()

markers = {
    "/Font": "font references (text layer present)",
    "/Image": "image XObjects",
    "/DCTDecode": "JPEG-compressed images (scan indicator)",
    "/CCITTFaxDecode": "fax/bitonal scans (scan indicator)",
    "/JPXDecode": "JPEG2000 images (scan indicator)",
}
for marker, meaning in markers.items():
    print(f"{marker:18s} {raw.count(marker.encode()):5d}   {meaning}")

/Font                142   font references (text layer present)
/Image               115   image XObjects
/DCTDecode             0   JPEG-compressed images (scan indicator)
/CCITTFaxDecode        0   fax/bitonal scans (scan indicator)
/JPXDecode             0   JPEG2000 images (scan indicator)


### 1b. Per-page structure

Document-wide counts confirm the file is born-digital, but further checks are required about the
individual pages being read. A page that is dense prose and a page that is a ruled
table need different things from the prompt, so there is a need to profile them separately.

**Reading the columns:** `num-hvy` counts lines carrying six or more digits — a
proxy for data rows, since sentences rarely reach that. Its ratio to total lines
is the prose/table discriminator. `vector` counts drawing operations: table rules
and chart geometry.

In [ ]:
# Per-page structural profile of the pages Parts 1-2 actually read.
# Pypdf exposes text, while PyMyPDF exposes graphics and fonts

import pymupdf
import pypdf

TARGETS = {
    1: "Part 2 - distribution date",
    5: "Part 1 - CIT, YoY, tax list",
    6: "Part 1 - tax list (cont.)",
    8: "Part 1 - Table 1.1, fiscal position",
    9: "chart page (control)", #not in spec, checks if charts are recoverable
    16: "Table 2.1 - FY2024 figures", #not in spec, only if spec requirement asks for FY2024
    20: "Part 1 - Table 2.4, top-ups",
    36: "Part 2 - estate duty date",
}

reader = pypdf.PdfReader(str(PDF))
doc = pymupdf.open(str(PDF))

print(f"{'page':>4} {'chars':>6} {'lines':>6} {'num-hvy':>8} {'raster':>7} {'vector':>7} {'fonts':>6}  shape")
print("-" * 86)
for page_no, purpose in TARGETS.items():
    page = doc[page_no - 1] #dicts are 1-indexed, libraries are 0-i ndexed
    text = reader.pages[page_no - 1].extract_text() or ""
    lines = [ln for ln in text.splitlines() if ln.strip()]
    # a line with >=6 digits is almost certainly a data row, not a sentence
    numeric = sum(1 for ln in lines if sum(ch.isdigit() for ch in ln) >= 6)
    ratio = numeric / len(lines) if lines else 0
    vectors = len(page.get_drawings())
    # ruled tables show BOTH dense numeric lines and many vector ops (the rules).
    # p.20 is the case that needs the vector test: its amounts are short (6,000 / 50 / 2)
    # so few lines clear the 6-digit bar, but 116 vector ops make it plainly a table.
    if ratio > 0.4 or vectors > 60:
        shape = "TABLE"
    elif len(lines) < 10:
        shape = "title/short"
    elif ratio > 0.15:
        shape = "prose+figures"
    else:
        shape = "prose"
    print(
        f"{page_no:>4} {len(text):>6} {len(lines):>6} {numeric:>8} "
        f"{len(page.get_images()):>7} {vectors:>7} {len(page.get_fonts()):>6}  {shape}"
    )
doc.close()

page  chars  lines  num-hvy  raster  vector  fonts  shape
--------------------------------------------------------------------------------------
   1    142      5        1       0       2      3  title/short
   5   2381     34        9       0       2      7  prose+figures
   6   2200     34        8       0       3      6  prose+figures
   8   3794     65       40       0     159      4  TABLE
   9    751     30        2       0      26      4  prose
  16   3605     69       36       0     124      4  TABLE
  20    474     16        1       0     116      3  TABLE
  36   3024     75        2       0       1      6  prose


## Criterion 2 — task requirements

### 2a. Row integrity on page 8 (Table 1.1)

The deciding test. The Corporate Income Tax row should read
`Corporate Income Tax 23.07 24.26 28.38 23.0 17.0` — three year columns then two
percentage columns. If a parser detaches the label from its figures, the LLM has to
guess which number belongs to which year.

In [4]:
import pdfplumber
import pymupdf
import pypdf


def text_pypdf(page: int) -> str:
    """Extract one 1-indexed page with pypdf."""
    return pypdf.PdfReader(str(PDF)).pages[page - 1].extract_text()


def text_pymupdf(page: int) -> str:
    """Extract one 1-indexed page with PyMuPDF."""
    with pymupdf.open(str(PDF)) as doc:
        return doc[page - 1].get_text()


def text_pdfplumber(page: int) -> str:
    """Extract one 1-indexed page with pdfplumber."""
    with pdfplumber.open(str(PDF)) as doc:
        return doc.pages[page - 1].extract_text() or ""


PARSERS = {"pypdf": text_pypdf, "PyMuPDF": text_pymupdf, "pdfplumber": text_pdfplumber}

In [5]:
def show_rows(page: int, needle: str) -> None:
    """Print every line containing `needle`, for each parser."""
    for name, fn in PARSERS.items():
        hits = [ln for ln in fn(page).splitlines() if needle.lower() in ln.lower()]
        print(f"--- {name} --- ({len(hits)} matching line(s))")
        for ln in hits:
            print(f"    {ln.strip()[:120]}")
        print()


show_rows(8, "Corporate Income Tax")

--- pypdf --- (1 matching line(s))
    Corporate Income Tax 23.07 24.26 28.38 23.0 17.0

--- PyMuPDF --- (1 matching line(s))
    Corporate Income Tax



--- pdfplumber --- (1 matching line(s))
    Corporate Income Tax 23.07 24.26 28.38 23.0 17.0



**Read the output above.** A parser passes if the label and all five figures land on
one line. It fails if the label appears alone, or the numbers appear without their
label — that is the fragmentation being tested for.

In [6]:
# Widen the check: does every target label keep its figures?
LABELS = [
    "OPERATING REVENUE",
    "Corporate Income Tax",
    "Personal Income Tax",
    "OVERALL FISCAL POSITION",
]

for name, fn in PARSERS.items():
    lines = fn(8).splitlines()
    print(f"--- {name} ---")
    for label in LABELS:
        row = next((ln.strip() for ln in lines if label.lower() in ln.lower()), None)
        has_digits = row is not None and any(ch.isdigit() for ch in row.replace(label, ""))
        print(f"  {'OK  ' if has_digits else 'FAIL'} {label:26s} {(row or '(not found)')[:80]}")
    print()

--- pypdf ---
  OK   OPERATING REVENUE          OPERATING REVENUE 91.01 96.70 104.30 14.6 7.9
  OK   Corporate Income Tax       Corporate Income Tax 23.07 24.26 28.38 23.0 17.0
  OK   Personal Income Tax        Personal Income Tax 15.52 16.84 17.53 12.9 4.1
  OK   OVERALL FISCAL POSITION    OVERALL FISCAL POSITION 1.72 (0.35) (3.57)

--- PyMuPDF ---
  FAIL OPERATING REVENUE          OPERATING REVENUE
  FAIL Corporate Income Tax       Corporate Income Tax
  FAIL Personal Income Tax        Personal Income Tax
  FAIL OVERALL FISCAL POSITION    OVERALL FISCAL POSITION



--- pdfplumber ---
  OK   OPERATING REVENUE          OPERATING REVENUE 91.01 96.70 104.30 14.6 7.9
  OK   Corporate Income Tax       Corporate Income Tax 23.07 24.26 28.38 23.0 17.0
  OK   Personal Income Tax        Personal Income Tax 15.52 16.84 17.53 12.9 4.1
  OK   OVERALL FISCAL POSITION    OVERALL FISCAL POSITION 1.72 (0.35) (3.57)



### 2b. Page 20 (Table 2.4) and page 5 (prose)

Page 20 holds the top-ups total. Page 5 is narrative text, where all three parsers
should agree — a control confirming differences are table-specific, not general.

In [7]:
print("=" * 25, "page 20 — Total row", "=" * 25)
show_rows(20, "Total")

print("=" * 25, "page 5 — prose control", "=" * 25)
show_rows(5, "Corporate Income Tax collections")

========================= page 20 — Total row =========================
--- pypdf --- (1 matching line(s))
    Total 20,352

--- PyMuPDF --- (1 matching line(s))
    Total

--- pdfplumber --- (1 matching line(s))
    Total 20,352

========================= page 5 — prose control =========================


--- pypdf --- (1 matching line(s))
    Corporate Income Tax collections are revised to $ 28.4 billion, which is

--- PyMuPDF --- (1 matching line(s))
    Corporate Income Tax collections are revised to $28.4 billion, which is

--- pdfplumber --- (1 matching line(s))
    Corporate Income Tax collections are revised to $28.4 billion, which is



### 2c. True cell extraction

`pdfplumber.extract_table()` returns a list of rows of cells rather than a line of
text. If it recovers Table 1.1 cleanly, that is a stronger guarantee than any
text-extraction result — the column a number sits in becomes explicit rather than
inferred from word order.

In [8]:
with pdfplumber.open(str(PDF)) as doc:
    tables = doc.pages[7].extract_tables()

print(f"tables found on page 8: {len(tables)}")
for t_idx, table in enumerate(tables):
    print(f"\ntable {t_idx}: {len(table)} rows x {max(len(r) for r in table)} cols")
    for row in table[:12]:
        print("   ", [(c or "").strip()[:24] for c in row])

tables found on page 8: 0


### 2d. Timing

Speed only matters as a tiebreak between parsers that are already correct.

In [9]:
import time

for name, fn in PARSERS.items():
    start = time.perf_counter()
    for _ in range(3):
        for page in PAGES:
            fn(page)
    elapsed = (time.perf_counter() - start) / 3
    print(f"{name:12s} {elapsed * 1000:7.1f} ms for {len(PAGES)} pages")

pypdf          443.3 ms for 4 pages
PyMuPDF         52.6 ms for 4 pages


pdfplumber     794.2 ms for 4 pages


### 2e. Images and charts

Pages 9, 10 and 17 carry charts. The question is whether any parser recovers the
*plotted values* — and whether the images are photographic (scans) or vector
drawings.

In [10]:
with pymupdf.open(str(PDF)) as doc:
    for page_no in (9, 10, 17):
        page = doc[page_no - 1]
        rasters = page.get_images(full=True)
        vectors = page.get_drawings()
        print(f"page {page_no:2d}: {len(rasters)} raster image(s), {len(vectors)} vector drawing(s)")

print("\nText extracted from the page 9 chart:")
chart_text = text_pymupdf(9)
print("   ", " ".join(chart_text.split())[:400])

page  9: 0 raster image(s), 26 vector drawing(s)
page 10: 0 raster image(s), 8 vector drawing(s)
page 17: 0 raster image(s), 9 vector drawing(s)

Text extracted from the page 9 chart:
    MINISTRY OF FINANCE 9 Chart 1.1 Breakdown of Government Operating Revenue in FY20231 Note: Figures may not add up to 100% due to rounding. 1 Total Revenue comprises Operating Revenue and NIRC. Operating Revenue, which includes tax and non-tax revenues (shown above), comprises 82.0% of Total Revenue in FY2023 while NIRC makes up the remaining 18.0%. Corporate Income Tax 27.2% Personal Income Tax 16


Chart *labels* extract as text because they are drawn as text. The plotted values
behind the wedges are vector geometry — not present as data in the file, so no
parser and no OCR recovers them. If a chart value is ever needed, read it from the
corresponding table instead.

## Criterion 3 — LangChain compatibility

Each candidate has a loader returning `Document` objects with page metadata, so the
choice does not constrain Part 3.

In [11]:
# Requires: uv add langchain-community
try:
    from langchain_community.document_loaders import (
        PDFPlumberLoader,
        PyMuPDFLoader,
        PyPDFLoader,
    )

    for loader_cls in (PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader):
        docs = loader_cls(str(PDF)).load()
        page_8 = next(d for d in docs if d.metadata.get("page") == 7)
        print(f"{loader_cls.__name__:20s} {len(docs):3d} docs   metadata keys: {sorted(page_8.metadata)[:5]}")
except ImportError as exc:
    print(f"skipped — {exc}")

/var/folders/n8/tln3gy6950zf139prl4z9cc40000gn/T/ipykernel_7971/732773868.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


PyPDFLoader           37 docs   metadata keys: ['creationdate', 'creator', 'moddate', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_actionid', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_contentbits']
PyMuPDFLoader         37 docs   metadata keys: ['author', 'creationDate', 'creationdate', 'creator', 'file_path']


PDFPlumberLoader      37 docs   metadata keys: ['CreationDate', 'MSIP_Label_153db910-0838-4c35-bb3a-1ee21aa199ac_ActionId', 'MSIP_Label_153db910-0838-4c35-bb3a-1ee21aa199ac_ContentBits', 'MSIP_Label_153db910-0838-4c35-bb3a-1ee21aa199ac_Enabled', 'MSIP_Label_153db910-0838-4c35-bb3a-1ee21aa199ac_Method']


## Verdict

Based on results, `pypdf` is chosen as the main parser to focus due to the following reasons.

### Scorecard

| Criterion | pypdf | PyMuPDF | pdfplumber |
|---|:---:|:---:|:---:|
| **Row integrity — p.8 Table 1.1** | Pass | Fail | Pass |
| **Row integrity — p.20 Table 2.4** | Pass | Fail | Pass |
| Prose — p.5 | Pass | Pass | Pass |
| `extract_table()` cell grid | n/a | n/a | Fail |
| Speed (4 pages) | 450 ms | **52 ms** | 823 ms |
| LangChain loader | Pass | Pass | Pass |

### Findings

| Criterion | Finding |
|---|---|
| 1. Nature of PDF | Born-digital. 142 font refs; **zero** DCTDecode / CCITTFaxDecode / JPXDecode; zero raster images on any target page. **OCR ruled out.** |
| 2a. Row integrity (p.8) | pypdf and pdfplumber return `Corporate Income Tax 23.07 24.26 28.38 23.0 17.0`. PyMuPDF returns the bare label, every figure detached — all 4 labels tested fail, incl. `OVERALL FISCAL POSITION`. |
| 2b. p.20 / p.5 | Same split on p.20 (`Total 20,352` vs bare `Total`). On p.5 prose all three agree, so the failure is **table-specific** — exactly where 3 of the 5 fields live. |
| 2c. Cell extraction | `extract_tables()` found **0 tables** on p.8: the table is drawn without ruling lines and detection is line-based. No parser yields a cell grid here. |
| 2d. Speed | PyMuPDF 9× faster than pypdf; pypdf 1.8× faster than pdfplumber. Only a tiebreak between parsers that are already correct. |
| 3. LangChain | All three loaders return 37 `Document` objects. **Not a discriminator** — any parser's output can be wrapped in a `Document` by hand. |

### Per-page structure

| Page | Shape | Evidence |
|---|---|---|
| 1 | title/short | 5 lines |
| 5, 6 | prose + figures | 9/34, 8/34 numeric-heavy lines — figures sit inside sentences |
| 8 | **TABLE** | 40/65 numeric, 159 vector ops |
| 9 | chart | 26 vector ops — labels are text, values are geometry |
| 16 | **TABLE** | 36/69 numeric, 124 vector ops |
| 20 | **TABLE** | 116 vector ops (short amounts, so few numeric lines) |
| 36 | prose | 2/75 numeric — glossary |

### Decision

**Chosen: `pypdf`, called directly.**

**Because:** it is one of the two parsers that keeps a table row intact, and the
faster of those two. Row integrity decides because three of the five fields are read
from Table 1.1, where a figure means nothing unless it stays bound to its label. 

**Rejected:**

- **PyMuPDF** — although it is fastest by far, it fragments every table row
  tested, which is not ideal in our case. 
- **pdfplumber** — identical text to pypdf but 1.8× slower. Furthermore, `extract_table()`
  capability was not able to extract any table (e.g. returned 0 tables at page 8, which should output a table). This is due to the table's rules being drawn as filled rectangles rather than stroked lines, so pdfplumber's default lines strategy finds nothing. Though there is a way to fix it, since `pypdf` gives correct rows, no further fixes are performed
- **Docling** — built for scanned or layout-complex documents; its ML layout analysis
  solves a problem this PDF does not have.
- **OCR** — zero raster images. Not applicable.

### Known limitation

**pypdf reads table *content*, not table *structure*.**

- Page 8 arrives as text: `28.38` sits on the right line, but nothing marks it as
  belonging to the *Revised FY2023* column.
- Column meaning lives in the header line elsewhere on the page. Only the LLM
  connects the two.
- Acceptable here: the prompt supplies header and data row together and instructs
  matching on headers, not position.
- Not acceptable if headers were absent, ambiguous, or split across lines.
- No fallback parser helps — pdfplumber's `extract_tables()` also returns nothing
  on this page (see Rejected, above).

**Chart values are not recoverable by any parser.**

- Pages 9, 10 and 17 hold 0 raster images and only vector drawings.
- Chart labels extract as text; the plotted values are geometry, not data.
- Not a parser limitation but a document one — OCR does not help either.
- No impact on Part 1: all five fields have a table or prose source. Relevant only
  if a later question points at a figure that exists solely in a chart.

**Charts**

- Pages 9, 10 and 17 hold **0 raster images** and only vector drawings.
- Chart labels extract as text; the plotted values are geometry, not data.
- No parser and no OCR recovers them. Chart values exist only as vector geometry — the underlying numbers were never stored in the file, so neither a parser nor OCR can recover them. Read the corresponding table instead.

### Assumptions

- **The verdict is document-specific.** Parser behaviour depends on how a given PDF
  was produced; the same three libraries can rank differently on another file, or on
  a later edition of this one.
- **PyMuPDF is not disqualified in general.** It is widely recommended as the fastest
  default and loses here only on this file's table layout.
- **A new source warrants a fresh run.** Point the path in cell 1 at the new file,
  set the target pages to whatever that document's requirements cite, and re-run. The
  criteria stay the same; only the evidence changes.
- **Production code reads path and pages from `config.yml`,** so switching documents
  there is a config change, not a code change. The notebook keeps its path inline —
  it is an investigation, not part of the pipeline.

### Carried forward

- `src/ingestion/parser.py` calls `pypdf.PdfReader` directly.
- **The prompt must bind figures to columns by reading header text, never position.**
- Page 16 (Table 2.1, FY2024) profiles as a table like p.8, but its row integrity was
  **not** tested here — verify before relying on it if the FY2024 reading is adopted.